# Full Calculus Knowledge Graph Generator (Fixed)
## OpenStax Calculus Volumes 1, 2, and 3

Uses Claude API to generate concept nodes and typed edges for every chapter
across all three OpenStax Calculus volumes.

**Output:** `all_calculus_nodes.csv` and `all_calculus_edges.csv` for Cosmograph.

**Setup:**
1. Get an API key from https://console.anthropic.com
2. Paste it into the cell below where it says `YOUR_KEY_HERE`
3. Run all cells top to bottom

Author: Catherine Cho — Hanlon/Coulter Scholars

In [ ]:
%pip install anthropic pandas tqdm

In [ ]:
import anthropic
import json
import time
import os
import pandas as pd
from tqdm.auto import tqdm
from collections import Counter

# =============================================
# PASTE YOUR API KEY HERE
# =============================================
API_KEY = "YOUR_KEY_HERE"

MODEL = "claude-sonnet-4-5"
MAX_TOKENS = 3000  # generous to avoid truncated JSON

client = anthropic.Anthropic(api_key=API_KEY)

# Quick sanity check that the API key works
try:
    test = client.messages.create(
        model=MODEL,
        max_tokens=20,
        messages=[{"role": "user", "content": "say hi"}],
    )
    print("API key works. Model:", MODEL)
    print("Test response:", test.content[0].text)
except Exception as e:
    print("API key test FAILED:", e)
    print("Fix this before continuing.")

## Reset progress (run only if you want to start fresh)

If you want to resume from a previous run, skip this cell.
If you want to start completely from scratch, run it.

In [ ]:
# Uncomment to clear progress and start over
# if os.path.exists("kg_progress.json"):
#     os.remove("kg_progress.json")
#     print("Cleared progress file.")
# else:
#     print("No progress file to clear.")

## Full OpenStax Table of Contents — All 3 Volumes

In [ ]:
TEXTBOOK = {
    # VOLUME 1
    "V1-Ch1": {"title": "Functions and Graphs", "volume": 1, "chapter": 1,
        "sections": [
            "1.1 Review of Functions",
            "1.2 Basic Classes of Functions",
            "1.3 Trigonometric Functions",
            "1.4 Inverse Functions",
            "1.5 Exponential and Logarithmic Functions",
        ]},
    "V1-Ch2": {"title": "Limits", "volume": 1, "chapter": 2,
        "sections": [
            "2.1 A Preview of Calculus",
            "2.2 The Limit of a Function",
            "2.3 The Limit Laws",
            "2.4 Continuity",
            "2.5 The Precise Definition of a Limit",
        ]},
    "V1-Ch3": {"title": "Derivatives", "volume": 1, "chapter": 3,
        "sections": [
            "3.1 Defining the Derivative",
            "3.2 The Derivative as a Function",
            "3.3 Differentiation Rules",
            "3.4 Derivatives as Rates of Change",
            "3.5 Derivatives of Trigonometric Functions",
            "3.6 The Chain Rule",
            "3.7 Derivatives of Inverse Functions",
            "3.8 Implicit Differentiation",
            "3.9 Derivatives of Exponential and Logarithmic Functions",
        ]},
    "V1-Ch4": {"title": "Applications of Derivatives", "volume": 1, "chapter": 4,
        "sections": [
            "4.1 Related Rates",
            "4.2 Linear Approximations and Differentials",
            "4.3 Maxima and Minima",
            "4.4 The Mean Value Theorem",
            "4.5 Derivatives and the Shape of a Graph",
            "4.6 Limits at Infinity and Asymptotes",
            "4.7 Applied Optimization Problems",
            "4.8 L'Hopital's Rule",
            "4.9 Newton's Method",
            "4.10 Antiderivatives",
        ]},
    "V1-Ch5": {"title": "Integration", "volume": 1, "chapter": 5,
        "sections": [
            "5.1 Approximating Areas",
            "5.2 The Definite Integral",
            "5.3 The Fundamental Theorem of Calculus",
            "5.4 Integration Formulas and the Net Change Theorem",
            "5.5 Substitution",
            "5.6 Integrals Involving Exponential and Logarithmic Functions",
            "5.7 Integrals Resulting in Inverse Trigonometric Functions",
        ]},
    "V1-Ch6": {"title": "Applications of Integration", "volume": 1, "chapter": 6,
        "sections": [
            "6.1 Areas Between Curves",
            "6.2 Determining Volumes by Slicing",
            "6.3 Volumes of Revolution: Cylindrical Shells",
            "6.4 Arc Length of a Curve and Surface Area",
            "6.5 Physical Applications",
            "6.6 Moments and Centers of Mass",
            "6.7 Integrals, Exponential Functions, and Logarithms",
            "6.8 Exponential Growth and Decay",
            "6.9 Calculus of the Hyperbolic Functions",
        ]},
    # VOLUME 2
    "V2-Ch1": {"title": "Integration Techniques", "volume": 2, "chapter": 1,
        "sections": [
            "1.1 Integration by Parts",
            "1.2 Trigonometric Integrals",
            "1.3 Trigonometric Substitution",
            "1.4 Partial Fractions",
            "1.5 Other Strategies for Integration",
            "1.6 Numerical Integration",
            "1.7 Improper Integrals",
        ]},
    "V2-Ch2": {"title": "Introduction to Differential Equations", "volume": 2, "chapter": 2,
        "sections": [
            "2.1 Basics of Differential Equations",
            "2.2 Direction Fields and Numerical Methods",
            "2.3 Separable Equations",
            "2.4 The Logistic Equation",
            "2.5 First-order Linear Equations",
        ]},
    "V2-Ch3": {"title": "Sequences and Series", "volume": 2, "chapter": 3,
        "sections": [
            "3.1 Sequences",
            "3.2 Infinite Series",
            "3.3 The Divergence and Integral Tests",
            "3.4 Comparison Tests",
            "3.5 Alternating Series",
            "3.6 Ratio and Root Tests",
        ]},
    "V2-Ch4": {"title": "Power Series", "volume": 2, "chapter": 4,
        "sections": [
            "4.1 Power Series and Functions",
            "4.2 Properties of Power Series",
            "4.3 Taylor and Maclaurin Series",
            "4.4 Working with Taylor Series",
        ]},
    # VOLUME 3
    "V3-Ch1": {"title": "Parametric Equations and Polar Coordinates", "volume": 3, "chapter": 1,
        "sections": [
            "1.1 Parametric Equations",
            "1.2 Calculus of Parametric Curves",
            "1.3 Polar Coordinates",
            "1.4 Area and Arc Length in Polar Coordinates",
            "1.5 Conic Sections",
        ]},
    "V3-Ch2": {"title": "Vectors in Space", "volume": 3, "chapter": 2,
        "sections": [
            "2.1 Vectors in the Plane",
            "2.2 Vectors in Three Dimensions",
            "2.3 The Dot Product",
            "2.4 The Cross Product",
            "2.5 Equations of Lines and Planes in Space",
            "2.6 Quadric Surfaces",
            "2.7 Cylindrical and Spherical Coordinates",
        ]},
    "V3-Ch3": {"title": "Vector-Valued Functions", "volume": 3, "chapter": 3,
        "sections": [
            "3.1 Vector-Valued Functions and Space Curves",
            "3.2 Calculus of Vector-Valued Functions",
            "3.3 Arc Length and Curvature",
            "3.4 Motion in Space",
        ]},
    "V3-Ch4": {"title": "Differentiation of Functions of Several Variables", "volume": 3, "chapter": 4,
        "sections": [
            "4.1 Functions of Several Variables",
            "4.2 Limits and Continuity (Multivariable)",
            "4.3 Partial Derivatives",
            "4.4 Tangent Planes and Linear Approximations",
            "4.5 The Chain Rule (Multivariable)",
            "4.6 Directional Derivatives and the Gradient",
            "4.7 Maxima/Minima Problems (Multivariable)",
            "4.8 Lagrange Multipliers",
        ]},
    "V3-Ch5": {"title": "Multiple Integration", "volume": 3, "chapter": 5,
        "sections": [
            "5.1 Double Integrals over Rectangular Regions",
            "5.2 Double Integrals over General Regions",
            "5.3 Double Integrals in Polar Coordinates",
            "5.4 Triple Integrals",
            "5.5 Triple Integrals in Cylindrical and Spherical Coordinates",
            "5.6 Calculating Centers of Mass and Moments of Inertia",
            "5.7 Change of Variables in Multiple Integrals",
        ]},
    "V3-Ch6": {"title": "Vector Calculus", "volume": 3, "chapter": 6,
        "sections": [
            "6.1 Vector Fields",
            "6.2 Line Integrals",
            "6.3 Conservative Vector Fields",
            "6.4 Green's Theorem",
            "6.5 Divergence and Curl",
            "6.6 Surface Integrals",
            "6.7 Stokes' Theorem",
            "6.8 The Divergence Theorem",
        ]},
    "V3-Ch7": {"title": "Second-Order Differential Equations", "volume": 3, "chapter": 7,
        "sections": [
            "7.1 Second-Order Linear Equations",
            "7.2 Nonhomogeneous Linear Equations",
            "7.3 Applications (Second-Order ODEs)",
            "7.4 Series Solutions of Differential Equations",
        ]},
}

total_sections = sum(len(ch["sections"]) for ch in TEXTBOOK.values())
print(f"Chapters: {len(TEXTBOOK)}")
print(f"Total sections: {total_sections}")

## Generation function

Robust error handling: retries on JSON errors and API errors with exponential backoff.
Tries to recover truncated JSON if Claude's response gets cut off.

In [ ]:
SYSTEM_PROMPT = """You are building a pedagogical knowledge graph for OpenStax Calculus.
Given a textbook section, generate concept nodes and typed edges.

Respond with ONLY valid JSON, no markdown fences, no extra text.
Keep your response compact — under 2500 tokens.

Format:
{
  "concepts": [
    {
      "name": "Short specific concept name (3-6 words)",
      "description": "One sentence definition",
      "hint": "One actionable sentence for a stuck student",
      "difficulty": 1
    }
  ],
  "edges": [
    {
      "source": "concept name exactly as above",
      "target": "concept name exactly as above OR a prior concept",
      "type": "prerequisite",
      "reason": "One sentence why"
    }
  ]
}

Rules:
- 5 to 10 concepts per section (not more — keep response compact)
- Name concepts specifically: 'Product Rule' not 'Differentiation'
- Hints must be actionable
- Edge types: "prerequisite" | "uses" | "related_to"
- Only add cross-section edges to DIRECT prerequisites
- Be conservative with edges — most pairs need none"""


def try_repair_json(text):
    """Attempt to fix common truncation issues."""
    text = text.strip()
    text = text.replace("```json", "").replace("```", "").strip()

    # If already valid, return
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try to find the last complete object/array and trim there
    # Walk backward through the string looking for a valid truncation point
    for end in range(len(text), 0, -1):
        candidate = text[:end]
        # Try closing brackets/braces in order
        for closing in ["", "}", "]}", "]}}", "}]}}", "\"]}}", "\"}]}}"]:
            try:
                return json.loads(candidate + closing)
            except json.JSONDecodeError:
                continue

        # If we're more than 200 chars from the original, give up trimming further
        if len(text) - end > 500:
            break

    return None


def generate_section(section_name, chapter_title, volume, prior_concepts, retries=3):
    """Ask Claude to generate concepts and edges for one section. Robust to errors."""
    prior_sample = prior_concepts[-50:] if len(prior_concepts) > 50 else prior_concepts
    prior_str = ", ".join(prior_sample) if prior_sample else "none yet"

    user_msg = f"""Section: {section_name}
Chapter: {chapter_title} (Volume {volume})

Recent prior concepts (you may reference these in edges):
{prior_str}

Generate concepts and edges. Keep JSON compact."""

    last_text = ""
    for attempt in range(retries):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_msg}],
            )
            last_text = response.content[0].text.strip()

            # Try strict parse first
            try:
                parsed = json.loads(last_text.replace("```json", "").replace("```", "").strip())
                return parsed
            except json.JSONDecodeError:
                # Try to repair
                parsed = try_repair_json(last_text)
                if parsed is not None and "concepts" in parsed:
                    print(f"    Recovered partial JSON ({len(parsed.get('concepts', []))} concepts)")
                    return parsed
                raise json.JSONDecodeError("unrecoverable", last_text, 0)

        except json.JSONDecodeError as e:
            if attempt < retries - 1:
                print(f"    JSON error (attempt {attempt+1}), retrying...")
                time.sleep(1)
            else:
                print(f"    Failed to parse JSON after {retries} attempts")
                print(f"    Last response (first 200 chars): {last_text[:200]}")
                return {"concepts": [], "edges": []}
        except Exception as e:
            print(f"    API error (attempt {attempt+1}): {str(e)[:200]}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                return {"concepts": [], "edges": []}

    return {"concepts": [], "edges": []}


print("Generation function ready.")

## Run the generator

Auto-resumes from `kg_progress.json` if it exists.
Also auto-retries sections that produced 0 concepts (likely failed last time).

In [ ]:
PROGRESS_FILE = "kg_progress.json"

# Load existing progress
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, "r") as f:
        progress = json.load(f)
    # Find sections that previously failed (produced no concepts) and remove them so they retry
    successful_sections = set()
    sections_with_concepts = set(c.get("section", "") for c in progress["all_concepts"])
    for s in progress["completed_sections"]:
        section_name = s.split("::")[-1]
        if section_name in sections_with_concepts:
            successful_sections.add(s)
    failed_count = len(progress["completed_sections"]) - len(successful_sections)
    progress["completed_sections"] = list(successful_sections)
    print(f"Resuming: {len(progress['all_concepts'])} concepts, {len(progress['all_edges'])} edges")
    if failed_count > 0:
        print(f"  Will retry {failed_count} previously failed section(s)")
else:
    progress = {
        "completed_sections": [],
        "all_concepts": [],
        "all_edges": [],
    }
    print("Starting fresh.")

completed = set(progress["completed_sections"])
all_concepts = progress["all_concepts"]
all_edges = progress["all_edges"]
prior_concept_names = [c["name"] for c in all_concepts]

for ch_key, ch_data in TEXTBOOK.items():
    vol = ch_data["volume"]
    ch_title = ch_data["title"]
    print(f"\n{'='*60}")
    print(f"Vol {vol} — {ch_title}")
    print(f"{'='*60}")

    for section in tqdm(ch_data["sections"], desc=ch_title[:30]):
        section_key = f"{ch_key}::{section}"
        if section_key in completed:
            continue

        result = generate_section(
            section_name=section,
            chapter_title=ch_title,
            volume=vol,
            prior_concepts=prior_concept_names,
        )

        for c in result.get("concepts", []):
            c["section"] = section
            c["chapter"] = ch_title
            c["volume"] = vol
            c["chapter_key"] = ch_key
            all_concepts.append(c)
            prior_concept_names.append(c["name"])

        for e in result.get("edges", []):
            e["section"] = section
            e["chapter"] = ch_title
            e["volume"] = vol
            all_edges.append(e)

        completed.add(section_key)
        n_c = len(result.get("concepts", []))
        n_e = len(result.get("edges", []))
        print(f"  {section}: {n_c} concepts, {n_e} edges")

        # Save progress every section
        progress["completed_sections"] = list(completed)
        progress["all_concepts"] = all_concepts
        progress["all_edges"] = all_edges
        with open(PROGRESS_FILE, "w") as f:
            json.dump(progress, f)

        time.sleep(0.5)

print(f"\n{'='*60}\nDONE")
print(f"Concepts: {len(all_concepts)}")
print(f"Edges: {len(all_edges)}")

## Add cross-volume prerequisite edges

The LLM misses obvious cross-volume connections — we add the most important ones here.

In [ ]:
# Cross-volume edges. Names use the same loose form Claude tends to generate;
# we'll resolve them to actual concept names by best match.
CROSS_VOLUME = [
    # V1 → V2 Integration Techniques
    ("u-substitution", "integration by parts", "prerequisite", "IBP follows u-sub as the next integration technique"),
    ("product rule", "integration by parts", "related_to", "IBP is the reverse of the product rule"),
    ("trig identities", "trigonometric integrals", "prerequisite", "Trig integrals require identities to simplify"),
    ("u-substitution", "trigonometric substitution", "prerequisite", "Trig sub is a specialized substitution"),
    ("rational functions", "partial fractions", "related_to", "Partial fractions decomposes rational functions"),
    ("definite integral", "improper integrals", "prerequisite", "Improper integrals extend definite integrals"),
    ("limits at infinity", "improper integrals", "prerequisite", "Improper integrals are defined as limits"),
    ("riemann sums", "numerical integration", "prerequisite", "Numerical methods generalize Riemann sums"),
    # V1 → V2 ODEs
    ("antiderivatives", "separable equations", "prerequisite", "Solving separable ODEs requires antidifferentiation"),
    ("exponential growth", "logistic equation", "prerequisite", "Logistic modifies exponential growth"),
    # V1 → V2 Series
    ("limits at infinity", "sequences", "prerequisite", "Sequence convergence is a limit as n → ∞"),
    ("improper integrals", "integral test", "prerequisite", "Integral test uses improper integrals"),
    ("derivative at a point", "taylor series", "prerequisite", "Taylor series uses all derivatives at a point"),
    # V1/V2 → V3 Parametric & Polar
    ("chain rule", "calculus of parametric curves", "prerequisite", "dy/dx for parametric uses chain rule"),
    ("arc length", "arc length in polar coordinates", "prerequisite", "Polar arc length extends Cartesian"),
    ("area between curves", "area in polar coordinates", "prerequisite", "Polar area uses integration"),
    # V1 → V3 Multivariable
    ("continuity", "limits and continuity", "prerequisite", "Multivariable continuity generalizes single-variable"),
    ("derivative function", "partial derivatives", "prerequisite", "Partial derivatives hold other variables fixed"),
    ("tangent line", "tangent planes", "related_to", "Tangent plane is 2D generalization of tangent line"),
    ("chain rule", "multivariable chain rule", "prerequisite", "Multivariable chain rule extends the original"),
    ("optimization", "lagrange multipliers", "prerequisite", "Lagrange handles constrained optimization"),
    ("critical points", "maxima/minima problems", "prerequisite", "Critical points generalize to multiple variables"),
    # V1 → V3 Multiple Integration
    ("definite integral", "double integrals", "prerequisite", "Double integrals extend single-variable"),
    ("polar coordinates", "double integrals in polar", "prerequisite", "Need polar before integrating in polar"),
    # V1 → V3 Vector Calculus
    ("fundamental theorem of calculus", "green's theorem", "related_to", "Green's is a 2D generalization of FTC"),
    ("fundamental theorem of calculus", "stokes' theorem", "related_to", "Stokes' generalizes FTC to surfaces"),
    ("fundamental theorem of calculus", "divergence theorem", "related_to", "Div theorem generalizes FTC to volumes"),
    ("partial derivatives", "gradient", "prerequisite", "Gradient is built from partial derivatives"),
    ("partial derivatives", "divergence and curl", "prerequisite", "Div and curl use partial derivatives"),
    # V2 → V3
    ("power series", "series solutions of differential equations", "prerequisite", "Series solutions are power series"),
]

# Build a lookup from lowercased concept name → real concept name
name_lookup = {c["name"].lower(): c["name"] for c in all_concepts}

def find_match(loose_name):
    """Find a concept whose name contains the loose name (case-insensitive)."""
    loose = loose_name.lower()
    # Exact match
    if loose in name_lookup:
        return name_lookup[loose]
    # Substring match
    for key, val in name_lookup.items():
        if loose in key or key in loose:
            return val
    return None

added = 0
unmatched = []
for src_loose, tgt_loose, etype, reason in CROSS_VOLUME:
    src = find_match(src_loose)
    tgt = find_match(tgt_loose)
    if src and tgt and src != tgt:
        all_edges.append({
            "source": src, "target": tgt, "type": etype,
            "reason": reason, "section": "cross-volume",
            "chapter": "cross-volume", "volume": 0,
        })
        added += 1
    else:
        unmatched.append((src_loose, tgt_loose, src, tgt))

print(f"Added {added} cross-volume edges")
if unmatched:
    print(f"Could not match {len(unmatched)} cross-volume edges:")
    for s, t, sm, tm in unmatched[:10]:
        print(f"  {s} -> {t}   (matched: {sm} -> {tm})")
print(f"Total edges now: {len(all_edges)}")

## Deduplicate

In [ ]:
# Dedupe concepts by name (keep first)
seen_names = set()
unique_concepts = []
for c in all_concepts:
    name = c["name"].strip()
    if name and name not in seen_names:
        seen_names.add(name)
        c["name"] = name
        unique_concepts.append(c)

# Dedupe edges; drop edges with missing endpoints
seen_edges = set()
unique_edges = []
dropped = 0
for e in all_edges:
    src = (e.get("source") or "").strip()
    tgt = (e.get("target") or "").strip()
    if not src or not tgt or src == tgt:
        dropped += 1
        continue
    if src not in seen_names or tgt not in seen_names:
        dropped += 1
        continue
    key = (src, tgt, e.get("type", ""))
    if key in seen_edges:
        continue
    seen_edges.add(key)
    e["source"] = src
    e["target"] = tgt
    unique_edges.append(e)

print(f"Unique concepts: {len(unique_concepts)}")
print(f"Unique edges: {len(unique_edges)}")
print(f"Dropped edges: {dropped}")

edge_types = Counter(e["type"] for e in unique_edges)
for t, c in edge_types.items():
    print(f"  {t}: {c}")

## Export to CSV for Cosmograph

In [ ]:
nodes_df = pd.DataFrame([
    {
        "id": c["name"],
        "description": c.get("description", ""),
        "hint": c.get("hint", ""),
        "difficulty": c.get("difficulty", ""),
        "volume": c.get("volume", ""),
        "chapter": c.get("chapter", ""),
        "section": c.get("section", ""),
    } for c in unique_concepts
])

edges_df = pd.DataFrame([
    {
        "source": e["source"],
        "target": e["target"],
        "type": e.get("type", ""),
        "reason": e.get("reason", ""),
        "volume": e.get("volume", ""),
    } for e in unique_edges
])

nodes_df.to_csv("all_calculus_nodes.csv", index=False)
edges_df.to_csv("all_calculus_edges.csv", index=False)

# Also save full JSON for the tutor app later
graph_json = {
    "concepts": {c["name"]: {
        "description": c.get("description", ""),
        "hint": c.get("hint", ""),
        "difficulty": c.get("difficulty", 1),
        "volume": c.get("volume", 1),
        "chapter": c.get("chapter", ""),
        "section": c.get("section", ""),
    } for c in unique_concepts},
    "edges": [{
        "source": e["source"], "target": e["target"],
        "type": e["type"], "reason": e.get("reason", ""),
    } for e in unique_edges]
}
with open("all_calculus_graph.json", "w") as f:
    json.dump(graph_json, f, indent=2)

print(f"Saved all_calculus_nodes.csv — {len(nodes_df)} nodes")
print(f"Saved all_calculus_edges.csv — {len(edges_df)} edges")
print(f"Saved all_calculus_graph.json\n")

print("Nodes by volume:")
for vol, grp in nodes_df.groupby("volume"):
    print(f"  Volume {vol}: {len(grp)} concepts")

print("\nTop 15 most-connected concepts:")
mentions = pd.concat([
    edges_df[["source"]].rename(columns={"source": "name"}),
    edges_df[["target"]].rename(columns={"target": "name"}),
])
top = mentions["name"].value_counts().head(15)
for name, count in top.items():
    print(f"  {count:4d}  {name}")